# Classification Capstone: Predicting Customer Purchasing Lapse
## Problem Framing

This project proposes predicting which recently active customers will place no valid merchandise order in the next 90 days. Its purpose is to help a retailer prioritize a limited, human-reviewed re-engagement campaign.

The framing below follows the supplied lecture notes and problem-framing assessment criteria. It defines the target, stakeholders, decision scenario, societal relevance, success criteria and boundaries. Numerical business targets are proposed assumptions, not achieved results or stakeholder-approved commitments.


# 1. Problem Definition & Context


## 1.1 Business problem and classification question

A retailer cannot give every customer the same amount of retention attention. Contacting everyone can waste staff time, offer unnecessary discounts and send unwanted messages. The CRM team needs a practical way to identify customers whose purchasing activity may be weakening.

> **At the start of each month, predict whether a customer who ordered in the previous 180 days will place no valid merchandise order in the next 90 days, using only information recorded before that date, so that the CRM team can prioritize customers for a limited re-engagement campaign.**

This follows the lecture's **five-part specification**:

| Part | Project definition |
|---|---|
| Decision | Which customers should the CRM team review for re-engagement? |
| Unit | One customer at one monthly prediction date |
| Event | No valid merchandise order during the following 90 days |
| Horizon | 90 days |
| Evidence | Customer purchase history available before the prediction date |

The proposed evidence is **UCI Online Retail II**, using both annual worksheets. It contains dated invoice lines, customer identifiers, products, quantities, prices and countries. These records can support customer-level histories and future purchase labels. The source is a UK-based online retailer with many wholesale customers; results cannot automatically be generalized to all retailers. [Dataset source and attribution: Chen, D. (2012), Online Retail II](https://doi.org/10.24432/C5CG6D).

A **valid merchandise order** must contain at least one genuine product line with a known customer, valid timestamp, positive quantity and positive price. Cancelled and adjustment invoices, exact duplicate lines, postage, fees, manual entries, discounts, vouchers and test records are excluded. Unusual product codes require review. Multiple product lines do not turn one invoice into several orders, and a postage-only invoice cannot establish a return purchase. The same rules apply to history and future outcomes. A later return does not retroactively change what was known at prediction time.

This is **binary classification**, because every eligible, fully observed customer has one of two outcomes. It is not regression: the target is a category, not the amount a customer will spend.


## 1.2 Target and eligible customers

The target is **lapse_90d**:

| Class | Definition | Decision meaning |
|---|---|---|
| **1 — Purchasing lapse** | No valid merchandise order in the next 90 days | Positive class: consider for re-engagement |
| **0 — Return purchase** | At least one valid merchandise order in the next 90 days | Lower priority for this campaign |

A customer is eligible if their Customer ID is known and they placed at least one valid order during the **180 days before the prediction date**. A customer with only one previous order can still qualify. Customers with no identifiable history, or no order in that window, are outside this project's population.

For prediction date the prediction date, use:

- **Eligibility and main feature window:** from 180 days before the prediction date up to, but not including, that date.
- **Outcome window:** from the prediction date up to, but not including, 90 days later.
- **Complete observation:** use a labelled example only when the entire outcome window is available.

An order exactly at the prediction date belongs to the outcome window. An order exactly at 90 days after the prediction date does not. An incomplete future window is **unlabelled**, not class 1. Each customer and prediction-date combination must occur once with exactly one label.

“Purchasing lapse” means no observed qualifying order at this retailer during the chosen period. It does **not** prove permanent churn, dissatisfaction or that a customer will respond to marketing.


In [1]:
# Five-part problem specification, adapted from the lecture notebook.
problem_spec = {
    "decision": "prioritize customers for human-reviewed re-engagement",
    "unit": "customer at a monthly prediction date",
    "event": "no valid merchandise order in the next 90 days",
    "horizon_days": 90,
    "prediction_time": "00:00 on the first day of each month",
    "evidence": "purchase history recorded before the prediction date",
    "eligibility": "at least one valid order in the previous 180 days",
    "target": "lapse_90d",
    "positive_class": "1 = purchasing lapse; 0 = return purchase",
}

for item, definition in problem_spec.items():
    print(f"{item}: {definition}")


decision: prioritize customers for human-reviewed re-engagement
unit: customer at a monthly prediction date
event: no valid merchandise order in the next 90 days
horizon_days: 90
prediction_time: 00:00 on the first day of each month
evidence: purchase history recorded before the prediction date
eligibility: at least one valid order in the previous 180 days
target: lapse_90d
positive_class: 1 = purchasing lapse; 0 = return purchase


## 1.3 Why this target fits the action

**Why 90 days?** A 30-day window can flag healthy customers who normally order every six to eight weeks. A 90-day window better fits a quarterly re-engagement objective and gives the team time to act. Some seasonal buyers will still be labelled as lapses, so the label should not be described as permanent churn.

**Why require activity within 180 days?** Including everyone who ever purchased would add long-inactive customers who are easy to classify but less useful for retention. The activity rule reduces that problem while keeping less frequent buyers. Customers last active 91–180 days ago still qualify, so later evaluation should compare performance across recency bands.

**Why use both worksheets?** The additional history supports a 180-day lookback, a full 90-day outcome window and later-time evaluation. It also provides some previous autumn behaviour before an autumn test. Two years remain a limited basis for generalizing across seasons.

The supplied feedback reports that approximately 83% of completed purchase gaps were 90 days or less. This is **preliminary supporting evidence from the feedback**, not a result reproduced here. Completed gaps describe customers who returned and omit customers who never returned; they cannot by themselves establish the best horizon.


## 1.4 Prediction timeline and leakage prevention

**Past 180 days:** determine eligibility and summarize purchase history.

**Prediction date:** estimate lapse risk and prepare the review shortlist.

**Following 90 days:** observe whether a qualifying purchase occurs and assign the outcome.

For example, a prediction on **1 June 2011** uses the main historical window from **3 December 2010 up to, but not including, 1 June 2011**. Its outcome window ends on **30 August 2011 at 00:00**, exclusive.

Features must use only information available before the prediction date. Future purchases and future returns cannot become inputs. The source timestamps do not establish when records reached the retailer's systems; timely availability is an offline assumption that must be checked before live use.


## 1.5 Hypothesis and candidate features

**Hypothesis:** past purchase frequency, spending and basket behaviour can identify purchasing lapses more accurately than using only days since the last order.

The meaningful comparison is therefore a **recency rule**: the longer since the last order, the higher the estimated risk. If a model does not improve on that rule, the extra complexity is not justified.

| Feature | Simple definition before prediction | Reason for considering it |
|---|---|---|
| Recency | Days since the most recent valid order | Measures current inactivity |
| Frequency | Distinct invoices in the last 30, 90 and 180 days | Describes purchasing regularity |
| Spending | Total quantity multiplied by price in the last 90 and 180 days | Measures recent purchasing activity |
| Average order value | Historical spending divided by order count | Describes typical basket value |
| Product diversity | Distinct merchandise StockCode values in 180 days | Describes breadth of purchasing |
| Observed tenure | Days since the first purchase seen before prediction | Helps distinguish short and long observed histories |

Spending is positive merchandise purchase value, not profit or net revenue after returns. Observed tenure is not necessarily the true relationship age because records start partway through the retailer's history.

Customer ID and Invoice are grouping keys, not model features. Product descriptions will not be one-hot encoded. Country will initially be used for fairness checks rather than prediction. Historical return features can be considered later if their timing and definitions are reliable.

These are **predictive associations**. They do not show that changing a feature, or contacting a customer, will prevent a lapse.


## 1.6 Stakeholders and a concrete decision scenario

On each monthly prediction date, the CRM team receives a ranked list and reviews at most the **highest-risk 20%** of eligible customers. The 20% limit is a proposed capacity assumption, to be confirmed with the business.

**Example:** if 3,000 customers qualify, up to 600 enter review. Staff check contact permission, previous campaign contact and unresolved service issues. They may send a relevant service check-in, postpone contact or remove the customer from the campaign. A high score does not automatically authorize an offer.

The proposed contact limit is **one campaign contact per customer per 90 days**. Customers outside the shortlist retain normal service. The model must not automatically change prices, deny service or send repeated promotions. Consent and contact-history information are not in this dataset, so the analytical shortlist is not an approved contact list.

| Stakeholder | Expected benefit | Main concern | Responsibility |
|---|---|---|---|
| CRM team | More focused use of limited time | Over-reliance on scores | Review the shortlist and contact rules |
| Customers | More relevant support and fewer blanket promotions | Unwanted contact or pressure | Preferences and opt-outs must be respected |
| Retailer | More efficient retention spending | Wasted offers or missed opportunities | Set capacity, costs and pilot limits |
| Data analyst | A clear, reproducible decision process | Leakage or misleading performance | Check data, errors and model limitations |


## 1.7 Societal relevance and impact

The potential social value is a more responsible relationship between the retailer and its customers: limited, relevant support instead of repeated mass promotions. Where wholesale customers are small businesses, a reliable supplier relationship may also support continuity of their operations.

This offers a plausible link to **SDG 8**, particularly support for entrepreneurship and small-business growth. However, the dataset identifies neither business size nor employment outcomes. It would be incorrect to claim that all wholesale customers are SMEs or that this model creates jobs.

The impact should be tested through **incremental 90-day contribution margin per £ of campaign spend**, compared with a randomized no-contact control. Contribution margin means revenue less variable product costs; campaign costs must also be recorded. Complaint and opt-out rates should be monitored alongside economic outcomes. These measures are not available in the supplied transactions and must be collected during a future pilot.

No environmental benefit is claimed. Additional purchases can increase deliveries and consumption, and the data contain no emissions or waste measurements.

### Benefit and harm quadrant from the lecture

| | Direct effect | Indirect effect |
|---|---|---|
| **Benefit** | Customers receive relevant support; CRM uses time more efficiently | Potentially more stable supplier relationships for wholesale businesses |
| **Harm** | Customers receive unnecessary contact; retailer wastes discounts | Infrequent or overseas customers receive unequal attention; increased consumption |

### Initial risk scoring

Following the lecture, **risk = severity × likelihood**, with both scored from 1 to 5. These are planning judgments, not measured probabilities. Risks scoring 15 or more require mitigation before a pilot.

| Potential harm | Severity | Likelihood | Risk score | Mitigation and owner |
|---|---:|---:|---:|---|
| Repeated unwanted contact | 4 | 4 | 16 | CRM lead: check permission and enforce the 90-day contact limit |
| Unequal targeting of customer groups | 4 | 3 | 12 | Analyst: compare group errors and selection rates; maintain normal service for all customers |
| Disclosure or linkage of purchase histories | 5 | 3 | 15 | Data owner: restrict access, prevent re-identification and publish aggregate results |


## 1.8 Success metrics and error costs

All precision, recall and F1 measures refer to **class 1: purchasing lapse**.

| Measure | Proposed success criterion | Decision connection |
|---|---|---|
| **Average precision (AP)** | At least **0.05 absolute improvement** over the recency rule on validation | The model should rank lapsing customers better than a simple rule |
| **Precision at the top 20%** | At least **0.05 absolute improvement** over recency at the same capacity | A fixed review workload should identify more actual lapses |
| Final test | Freeze all choices and check the same improvements on the later test | Establish whether the improvement carries forward in time |
| Pilot impact | Incremental contribution margin / campaign spend **greater than 1** | Additional contribution should exceed campaign cost |
| Customer treatment | No unauthorized contact or contact-limit breaches; complaint increase no more than **0.5 percentage points** versus control | Benefits must not depend on intrusive treatment |

These numbers are **proposed targets**, not achieved results or stakeholder-approved budgets. If they are missed, report the failure and reconsider whether the model is useful.

AP summarizes the precision–recall curve across score thresholds; it will be calculated with the standard average-precision calculation. It is not the same calculation as trapezoidal PR-AUC. Precision at the top 20% is the proportion of the shortlisted customers who actually lapse. Also report recall at that capacity, a confusion matrix, class-1 F1 and ROC-AUC. If scores are presented as probabilities, check calibration and Brier score.

| Error | What happens | Illustrative cost |
|---|---|---|
| False positive | Customer is flagged but would return; contact or discount may be unnecessary | £2 |
| False negative | Customer lapses without entering the shortlist; a support opportunity may be missed | £10 |

Under these assumptions, a false negative costs five times as much as a false positive. The average error penalty is **(£2 × false positives + £10 × false negatives) ÷ number of customers**. These are illustrative penalties, not measured losses: a missed lapse may not have been preventable. Repeat the analysis with equal costs and with a 10:1 ratio. Actual campaign economics must include the cost of every contact, including correct flags.

Start with the top-20% capacity rule, resolving tied scores reproducibly without using labels. If a probability threshold is used, select it on validation under the capacity limit and freeze it before testing; do not assume 0.5 is best. A high lapse probability does not imply a high probability of responding to an offer.


## 1.9 Validation plan

| Split | Monthly prediction dates | Latest outcome window ends |
|---|---|---|
| Training | June 2010 to March 2011 | 30 May 2011 |
| Validation | 1 June 2011 | 30 August 2011 |
| Test | 1 September 2011 | 30 November 2011 |

Use both sheets, subject to confirming these windows are fully covered. **Every training label must be known before the next evaluation date:** the training outcome window must end on or before the evaluation date. April and May 2011 training snapshots would violate this rule for June validation and must be excluded.

During training, use earlier expanding-time checks, for example December 2010 and March 2011, applying the same rule. Learn scaling, imputation and model settings from each training fold only. Use June validation to finalize the model and shortlist policy, then keep the fitted model fixed for the September test.

The capstone recommends stratification, but a random stratified split would mix past and future customer snapshots here. **Time-based evaluation is the justified primary choice.** Check class support in each split while preserving its natural prevalence. A customer-group split alone would also fail to preserve time order. A separate unseen-customer assessment could provide a secondary check.

The same customer can appear at multiple dates because the intended use is repeated scoring of existing customers. Those rows are not independent. Later uncertainty estimates should resample whole customers rather than individual rows. Report performance across time as well: one autumn test cannot establish year-round reliability.


## 1.10 Ethics and boundaries

The main concern is turning a prediction into repeated marketing pressure. Human review, permission checks and contact limits are therefore part of the decision design.

The data come from one retailer and an older period. Missing customer IDs exclude some purchases and may make the eligible population unrepresentative. Geography and purchasing patterns may also reflect unequal circumstances. Removing Country from model inputs does not remove all proxy bias.

Later checks should compare selection rates and errors for UK/non-UK customers, one-order/repeat-order customers and recency bands of 0–30, over 30–90 and over 90–180 days. Report group sizes; avoid firm conclusions for small groups. Missing demographic attributes mean demographic fairness cannot be established. Do not infer sensitive attributes or attempt re-identification.

The expected use is ordinary retail decision support and is provisionally considered lower-risk for project planning. This is not a formal legal classification. Any live deployment needs a review of applicable privacy and marketing rules. Public dataset availability is not marketing consent.

**In scope:** predicting a 90-day purchasing lapse for recently active customers. **Out of scope:** permanent churn, campaign uplift, product recommendations, automated contact, price discrimination and environmental-impact claims.


---
# 2. Dataset Selection & Description


## 2.1 Import Libraries

In [22]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2.2 Data Loading and Initial Understanding

The Online Retail II dataset is provided in two Excel worksheets covering consecutive periods: **2009–2010** and **2010–2011**. Both worksheets are loaded separately and concatenated into a single DataFrame to create a continuous transaction-level dataset for subsequent analysis.

Each row represents an individual product line within a customer invoice rather than a complete customer or order. Therefore, customer-level features and the final classification target will need to be constructed later by aggregating these transaction records over appropriate time windows.

In [3]:
# Load datasets
df1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name="Year 2009-2010")
df2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name="Year 2010-2011")

df = pd.concat([df1, df2], ignore_index=True)
display(df.shape)
df.head()

(1067371, 8)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Initial Dataset Overview

After combining the two worksheets, the dataset contains **1,067,371 transaction rows and 8 variables**.

The available variables describe invoice information, products, transaction quantities and prices, customer identifiers, transaction dates, and customer countries. The first observations confirm that the data is currently organised at the **invoice-product transaction level**.

The dataset covers transactions from **1 December 2009 to 9 December 2011**, providing approximately two years of purchasing history. This longitudinal structure is suitable for constructing historical customer behaviour features and future lapse outcomes using time-based prediction windows.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [5]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


### Initial Observations

The initial inspection highlights several characteristics that require further investigation before modelling:

- **Quantity** contains both positive and negative values, ranging from **-80,995 to 80,995**. Negative quantities may represent cancellations or returned products rather than ordinary purchases.
- **Price** also contains unusual values, including negative values and very large positive values. These records require investigation before deciding whether they represent valid transactions, adjustments, or data-quality issues.
- **Customer ID** is not available for every transaction, which is particularly important because the classification problem is defined at the customer level.
- The large ranges observed for Quantity and Price suggest that their distributions may be highly skewed and contain extreme observations.
- `InvoiceDate` is already stored as a datetime variable, which will support chronological splitting and construction of observation and outcome windows.

These observations are treated as issues for further investigation rather than immediately removing records, since unusual transaction values may have a legitimate business meaning.

### Understanding Data Types

The variables are reviewed by both their stored data type and their analytical
role. This distinction is important because a variable may be stored as numeric
without representing a continuous numerical feature. For example, `Customer ID`
is stored numerically but functions as an identifier rather than a quantitative
measurement.

Automatic type detection is therefore used as an initial diagnostic and will be
combined with domain understanding before preprocessing decisions are made.

In [21]:
def detect_variable_types(df, target_col=None):
    """Automatically detect variable types in a DataFrame"""
    types = {}
    
    for col in df.columns:
        if col == target_col:
            types[col] = 'target'
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            types[col] = 'datetime'
        elif pd.api.types.is_numeric_dtype(df[col]):
            if df[col].nunique() == 2:
                types[col] = 'binary'
            else:
                types[col] = 'numerical'
        elif df[col].nunique() < 20:  # Threshold for categorical
            types[col] = 'categorical'
        elif df[col].dtype == 'object':
            # Check if it's text (long strings) or categorical
            avg_length = df[col].astype(str).str.len().mean()
            if avg_length > 50:
                types[col] = 'text'
            else:
                types[col] = 'categorical'
    
    return types

detect_variable_types(df)

{'Invoice': 'categorical',
 'StockCode': 'categorical',
 'Description': 'categorical',
 'Quantity': 'numerical',
 'InvoiceDate': 'datetime',
 'Price': 'numerical',
 'Customer ID': 'numerical'}

### Data Quality Assessment

The initial data-quality assessment examines missing values, exact duplicate
records, zero values, numerical ranges, and transaction values that may violate
expected business rules.

At this stage, the purpose is diagnostic rather than corrective. Records are
flagged for investigation first, and cleaning decisions are made only after
considering their possible business meaning and their relevance to the
customer-level prediction problem.

In [23]:
# Basic missing value overview
def quick_missing_analysis(df):
    missing_counts = df.isnull().sum()
    missing_percent = (missing_counts / len(df)) * 100
    
    print("Missing Value Summary:")
    for col in df.columns:
        if missing_counts[col] > 0:
            print(f"  {col}: {missing_counts[col]} missing ({missing_percent[col]:.1f}%)")

quick_missing_analysis(df)

Missing Value Summary:
  Description: 4382 missing (0.4%)
  Customer ID: 243007 missing (22.8%)


#### Missing Values

Missingness is concentrated in two variables:

- `Description`: **4,382 missing records (0.4%)**
- `Customer ID`: **243,007 missing records (22.8%)**

Missing product descriptions affect only a small proportion of the data.
Missing customer identifiers are substantially more important because
customer-level purchase histories and lapse labels cannot be constructed
reliably for transactions without an identifiable customer.

The treatment of these records will therefore be addressed during the cleaning
stage rather than applying automatic imputation.

In [44]:
# Check duplicates
duplicates = df.duplicated().sum()
duplicates_percentage = (duplicates/len(df))*100
print(f"Exact duplicates: {duplicates} ({duplicates_percentage:.2f})%")

Exact duplicates: 34335 (3.22)%


#### Duplicate Records

The dataset contains **34,335 exact duplicate rows**, representing approximately
**3.22%** of all records.

Because these rows are exact duplicates, they may artificially inflate purchase
frequency, quantities, or customer spending if retained. Their treatment will
therefore be considered during the initial cleaning stage before customer-level
features are constructed.

In [39]:
# Check zero values in num_cols
def quick_zero_analysis(df):

  num_cols = df.select_dtypes(include=['number']).columns
  zero_analysis = []

  for col in num_cols:

    zero_analysis.append({
      'name': col,
      'zero_counts': len(df[df[col] == 0]),
      'zero_percentage': (len(df[df[col]==0])/len(df))*100
    })

  zero_df = pd.DataFrame(zero_analysis).round(2)

  return zero_df

quick_zero_analysis(df)

,name,zero_counts,zero_percentage
0,Quantity,0,0.00
1,Price,6202,0.58
2,Customer ID,0,0.00


In [ ]:
# Range and distribution check
def basic_range_check(df):
    numeric_cols = df.select_dtypes(include=['number']).columns
    
    for col in numeric_cols:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
        
        # Flag obviously problematic values
        if col.lower() in ['age'] and (df[col].min() < 0 or df[col].max() > 120):
            print(f"  Suspicious age values detected")
        elif col.lower() in ['salary', 'income', 'price'] and df[col].min() < 0:
            print(f"  Negative financial values detected")

basic_range_check(df)

Quantity: min=-80995, max=80995
Price: min=-53594.36, max=38970.0
  ⚠️ Negative financial values detected
Customer ID: min=12346.0, max=18287.0


In [62]:
# Domain-specific outlier detection
def check_business_rules(data):
    issues = []
    
    # Age should be reasonable
    if 'Quantity' in data.columns:
        bad_quantities = data[(data['Quantity'] < 0) | (data['Quantity'] > 120)]
        if len(bad_quantities) > 0:
            issues.append(f"Impossible quantities: {len(bad_quantities)} records")
    
    # Prices should be positive
    if 'Price' in data.columns:
        bad_prices = data[data['Price'] <= 0]
        if len(bad_prices) > 0:
            issues.append(f"Non-positive prices: {len(bad_prices)} records")
    
    return issues

check_business_rules(df)

['Impossible quantities: 31675 records', 'Non-positive prices: 6207 records']

#### Transaction Value Checks

The numerical checks identify several transaction patterns requiring further
investigation. Quantity contains negative values and substantial extreme values,
while Price includes zero, negative, and unusually large observations.

These values should not automatically be classified as outliers or errors.
In retail transaction data, negative quantities may correspond to product
returns or cancelled invoices, while unusual prices may include adjustments,
special transactions, or data-entry issues.

For this project, a **valid merchandise purchase** must therefore be explicitly
defined before cleaning. Transactions representing cancellations, returns,
adjustments, or other non-purchase activity should be distinguished from genuine
customer purchases before constructing behavioural features and the
`lapse_90d` target.

# 3. Initial Cleaning & Splitting


## 3.1 Initial cleaning

In [15]:
def initial_cleaning (df):

  # Remove exact duplicates
  df = df.copy()
  df = df.drop_duplicates()

  # Remove rows with Customer ID is missing
  df = df[df['Customer ID'].notna()]

  # Remove unrealistic value rows
  df = df[(df['Quantity'] > 0) | (df['Price'] > 0)]

  return df


In [16]:
df = initial_cleaning(df)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [13]:
df.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [14]:
df.duplicated().sum()

np.int64(0)

In [18]:
len(df[(df['Quantity'] < 0) | (df['Price'] < 0)])

0

# 4. Data imputing & EDA

*To be completed next.*


# 5. Preprocessing & Feature Engineering

*To be completed next.*


# 6. Model Development

*To be completed next.*


# 7. Model Evaluation, Optimization & Ethics

*To be completed after model development.*


# 8. Impact Reporting (Stakeholder-Facing)

*To be completed after final model selection.*
